In [8]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [9]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



# BB84, no attacker

BB84 between Alice and Bob, no eavesdropper. N = 100 qubits sent (~25-bit final key).

## Quantum RNG

In [10]:
from qiskit_aer import AerSimulator
simulator = AerSimulator()

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    return int(simulator.run(transpile(qc, simulator), shots=1, memory=True).result().get_memory()[0])

def quantum_random_bits(n, chunk=20):
    bits = []
    while n > 0:
        k = min(chunk, n)
        qc = QuantumCircuit(k, k)
        for q in range(k):
            qc.h(q)
        qc.measure(range(k), range(k))
        bs = simulator.run(transpile(qc, simulator), shots=1, memory=True).result().get_memory()[0]
        bits.extend(int(c) for c in bs[::-1])
        n -= k
    return bits

print("sample:", quantum_random_bits(10))

sample: [0, 1, 1, 1, 1, 1, 1, 0, 0, 1]


## Alice

In [11]:
N = 100

alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)   # 0 = standard (Z), 1 = diagonal (X)

def alice_encode(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

alice_circuits = [alice_encode(b, ba) for b, ba in zip(alice_bits, alice_bases)]

basis_name = {0: "Z", 1: "X"}
state_name = {(0, 0): "|0>", (1, 0): "|1>", (0, 1): "|+>", (1, 1): "|->"}

print(f"Alice prepared {N} qubits. First 10:")
print(f"{'i':>3} {'bit':>4} {'basis':>6} {'state':>6}")
for i in range(10):
    print(f"{i:>3} {alice_bits[i]:>4} {basis_name[alice_bases[i]]:>6} {state_name[(alice_bits[i], alice_bases[i])]:>6}")

Alice prepared 100 qubits. First 10:
  i  bit  basis  state
  0    1      Z    |1>
  1    1      Z    |1>
  2    1      Z    |1>
  3    1      X    |->
  4    1      Z    |1>
  5    0      X    |+>
  6    0      Z    |0>
  7    1      Z    |1>
  8    1      X    |->
  9    1      Z    |1>


## Bob

In [12]:
bob_bases = quantum_random_bits(N)

def bob_measure(prepared_circuit, basis):
    qc = prepared_circuit.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1, memory=True).result()
    return int(result.get_memory()[0])

bob_bits = [bob_measure(qc, ba) for qc, ba in zip(alice_circuits, bob_bases)]

print(f"Bob measured {N} qubits. First 10 results:")
print(f"{'i':>3} {'bob_basis':>10} {'bob_bit':>8}")
for i in range(10):
    print(f"{i:>3} {basis_name[bob_bases[i]]:>10} {bob_bits[i]:>8}")

Bob measured 100 qubits. First 10 results:
  i  bob_basis  bob_bit
  0          X        0
  1          Z        1
  2          X        1
  3          Z        0
  4          X        0
  5          X        0
  6          X        1
  7          Z        1
  8          X        1
  9          Z        1


## Sifting

In [13]:
matching_indices = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i] for i in matching_indices]
bob_sifted   = [bob_bits[i]   for i in matching_indices]

print(f"Out of {N} qubits, {len(matching_indices)} survived sifting "
      f"({100 * len(matching_indices) / N:.0f}%).")
print(f"Alice's sifted bits: {alice_sifted}")
print(f"Bob's   sifted bits: {bob_sifted}")

Out of 100 qubits, 50 survived sifting (50%).
Alice's sifted bits: [1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0]
Bob's   sifted bits: [1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0]


## Error check

In [14]:
THRESHOLD = 0.15

check_indices = list(range(0, len(alice_sifted), 2))
key_indices   = list(range(1, len(alice_sifted), 2))

alice_check = [alice_sifted[i] for i in check_indices]
bob_check   = [bob_sifted[i]   for i in check_indices]

mismatches = sum(1 for a, b in zip(alice_check, bob_check) if a != b)
error_rate = mismatches / len(alice_check) if alice_check else 0

alice_key = [alice_sifted[i] for i in key_indices]
bob_key   = [bob_sifted[i]   for i in key_indices]

print("=" * 60)
print("BB84 Summary: No Attacker")
print("=" * 60)
print(f"Qubits sent           : {N}")
print(f"Survived sifting      : {len(alice_sifted)}")
print(f"Check bits compared   : {len(alice_check)}")
print(f"Mismatches in check   : {mismatches}")
print(f"Estimated error rate  : {error_rate:.1%}")
print(f"Detection threshold   : {THRESHOLD:.0%}")
print()

if error_rate > THRESHOLD:
    print("ATTACK DETECTED. Aborting, key is not safe to use.")
else:
    print("No attack detected. Final shared key:")
    print(f"  Alice key : {alice_key}")
    print(f"  Bob   key : {bob_key}")
    print(f"  Match     : {alice_key == bob_key}")
    print(f"  Length    : {len(alice_key)} bits")

BB84 Summary: No Attacker
Qubits sent           : 100
Survived sifting      : 50
Check bits compared   : 25
Mismatches in check   : 0
Estimated error rate  : 0.0%
Detection threshold   : 15%

No attack detected. Final shared key:
  Alice key : [0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0]
  Bob   key : [0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0]
  Match     : True
  Length    : 25 bits


## Notes

Error rate is 0% because there's no attacker and no channel noise. The 15% threshold doesn't trigger here, it exists for the real-world case where channel noise is a few % and we still want to flag attackers (who push the rate to ~25%, see attacker notebook).